# Acquisition des données

## 0. Import de fonctions utiles à l'importation des données

Avant de procéder à l'acquisition des données, il est essentiel de configurer notre environnement de travail. Dans une optique de **reproductibilité** et de **clarté du code**, la logique technique (les fonctions de téléchargement) a été déportée dans un dossier dédié nommé `/fonctions`, composé de fichiers Python. 

La cellule suivante initialise cette connexion en ajoutant le dossier des scripts au chemin système de Python et en important l'ensemble des dépendances nécessaires.

In [3]:
import sys
import os
import soccerdata as sd
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
from concurrent.futures import ThreadPoolExecutor

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

# On importe toutes les fonctions dans le fichier imports.py
from imports import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# On définit la période sur laquelle on souhaite récolter les données
annee_debut = 2020
annee_fin = 2025

# On définit les ligues que l'on souhaite analyser
leagues = {"GB1", "ES1", "L1", "IT1", "FR1"}

# TODO : pour le README mettre les nomenclatures des ligues

## 1. Transfermarkt

Ce dataset est une extraction des données du site de référence mondial **Transfermarkt**, mise à disposition via le projet open-source `player-scores` de David Cariboo. Il s'agit d'une base de données combinant des faits observés (transferts, compositions) et des **estimations de marché** validées par un réseau d'experts. Les valeurs sont révisées périodiquement pour refléter l'offre et la demande réelle du football professionnel.


Transfermarkt est en fait la référence de base pour l'évaluation financière des joueurs. Dans ce projet, cette source est alors **essentielle** car elle fournit notre **variable cible** : la valeur marchande.


Les fichiers utilisés ont donc chacun leur utilité propre afin de prédire la valeur marchande des joueurs de football. Ils contiennent l'historique temporel des prix, constituant ce que le modèle devra prédire. De plus, nous avons accès aux caractéristiques personnelles des joueurs : âge, taille, pied fort, et la **date d'expiration du contrat**, variable critique pour l'évaluation financière. Concernant les performances sportives, ces données recensent chaque match disputé et le temps de jeu effectif (minutes), permettant de calculer le **volume d'activité** et l'importance réelle du joueur dans la rotation de son équipe. Les détails sur les recontres sont également disponibles. Cela permet de pondérer une performance individuelle selon le prestige ou la difficulté du match. Des informations sont également disponibles sur les clubs mais aussi les compétitions qui nous intéressent.


A partir de recherches bibliographiques précédemment réalisées, nous pouvons émettre quelques hypothèses sur ces données Transfermarkt.
Premièrement, on s'attend à un pic des valeurs marchandes entre 24 et 28 ans avant un déclin lié à la valeur de revente. De plus, un joueur approchant de la fin de son contrat devrait voir sa valeur marchande diminuer. Enfin, une baisse drastique des apparitions (titularisation ou même en tant que remplaçant) sans blessure devrait signaler une perte de valeur.

### **CONFIGURATION DE L'API KAGGLE**

Avant d'utiliser cette fonction, vous devez créer un fichier `.env` à la racine de votre projet avec le format suivant :

KAGGLE_USERNAME=votre_nom_d_utilisateur

KAGGLE_API_TOKEN=votre_cle_api

**Où trouver ces informations ?**

*   **Identifiant :** Il s'agit de votre nom d'utilisateur Kaggle habituel.
*   **Token :** Allez sur votre profil Kaggle > **Settings** > section **API** > Cliquez sur **"Generate New API Token"**. Copiez ensuite le token dans le fichier `.env`.

In [9]:
# Configuration pour utiliser la fonction de téléchargement de données kaggle
DATASET = 'davidcariboo/player-scores'
DEST = "../data/transfermarkt_datasets"

# Appel de la fonction de téléchargement
download_kaggle_dataset(DATASET, DEST)

Téléchargement de davidcariboo/player-scores vers ../data/transfermarkt_datasets...
Dataset URL: https://www.kaggle.com/datasets/davidcariboo/player-scores
Téléchargement correctement effectué !


In [10]:
# On conserve les joueurs du Big 5 dont la dernière saison est au moins après la saison de départ choisie
players_path = os.path.join(DEST, "players.csv")
appearances_path = os.path.join(DEST, "appearances.csv")

players_filtered(players_path, appearances_path, annee_debut, annee_fin, leagues)

players.csv filtré : 7125 joueurs conservés.


In [11]:
# On conserve les valeurs marchandes réalisées après le 01/07 de la première saison sélectionnée
valuations_path = os.path.join(DEST, "player_valuations.csv")

valuations_filtered(valuations_path, appearances_path, annee_debut, annee_fin, leagues)

player_valuations.csv filtré : 41152 lignes conservées.


Le dataset player-scores a été téléchargé avec succès. Les fichiers principaux (joueurs et valorisations) sont désormais disponibles pour l'étape d'exploration.

## 2. Soccerdata

Nous utilisons ici l'API de Soccerdata via un package Python pour récolter des données issues de deux sources majeures du football européen : FBref et Understat. FBref fournit en effet des données de performances complètes et est la source de référence pour les statistiques avancées. Understat est quant-à-lui un spécialiste des données d'**Expected Goals (xG)** et **Expected Assists (xA)**. Leurs modèles calculent la probabilité qu'un tir devienne un but en fonction de multiples facteurs contextuels (position, type de passe, etc.).


L'utilisation de ces sources combinées se justifie par plusieurs facteurs.
Tout d'abord, FBref offre un certain niveau de détailsur les actions défensives et la création de jeu (Progressive Carries, Tackles, etc.). De plus, Understat permet d'isoler la performance réelle de la "chance" ou de la "finition". Un joueur qui génère beaucoup de xG sans marquer reste une cible de transfert à fort potentiel. Enfin, l'API permet d'uniformiser les noms de ligues et les saisons entre les deux plateformes, réduisant le risque d'erreurs lors de la fusion.


Les variables ont été segmentées pour répondre aux besoins de notre futur modèle. Certaines évaluent le volume de jeu et l'influence tactique par poste tandis que d'autres mesurent la dangerosité et la contribution d'un joueur sur le terrain.


Enfin, nous pouvons supposer que les attaquants dont les `xG` sont élevés et constants affichent une valeur marchande supérieure, la capacité à se créer des occasions étant une compétence très valorisée sur le marché. Pour les créateurs, le `xA` est un meilleur prédicteur de la valeur que les passes décisives réelles, car il mesure la vision de jeu indépendamment du finisseur. Enfin, le championnat d'origine pourrait agir comme un coefficient multiplicateur sur la valeur marchande (prime liée à l'exposition financière de la Premier League par exemple).

In [25]:
# Nous récoltons ici les données d'Understat
df_xg = get_understat_xg(annee_debut, annee_fin, leagues)
df_xg.to_csv(r'..\data\soccerdata\data_xg_soccerdata.csv', index=False, sep=',', encoding='utf-8-sig')

Understat | Ligues : ['ITA-Serie A', 'ENG-Premier League', 'FRA-Ligue 1', 'GER-Bundesliga', 'ESP-La Liga'] | Saisons : [2020, 2021, 2022, 2023, 2024, 2025]


[5/11/2026 2:40:20 PM] INFO     Saving cached data to C:\Users\LouisHarle\soccerdata\data\Understat  _common.py:250

                       WARNING  c:\Users\LouisHarle\OneDrive -                                  ]8;id=13005557;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=13005558;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py#230\230]8;;\
                                Quadratic\Bureau\Stage_VM\.venv\Lib\site-packages\soccerdata\_c                    
                                ommon.py:144: UserWarning: Season id "2021" is ambiguous:                          
                                interpreting as "20-21"                                                            
                                  warnings.warn(msg, stacklevel=1)                                                 
                                                                                                                   

16601 lignes récupérées
Colonnes : ['league', 'season', 'team', 'player', 'league_id', 'season_id', 'team_id', 'player_id', 'position', 'matches', 'minutes', 'goals', 'xg', 'np_goals', 'np_xg', 'assists', 'xa', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'xg_chain', 'xg_buildup']


In [26]:
data_advanced = get_players_advanced_stats_range(annee_debut, annee_fin, leagues)
data_advanced.to_csv(r'..\data\soccerdata\data_soccerdata.csv', index=False, sep=',', encoding='utf-8-sig')

Ligues : ['ITA-Serie A', 'ENG-Premier League', 'FRA-Ligue 1', 'GER-Bundesliga', 'ESP-La Liga'] | Saisons : ['20-21', '21-22', '22-23', '23-24', '24-25', '25-26']


[5/11/2026 2:40:27 PM] INFO     Saving cached data to C:\Users\LouisHarle\soccerdata\data\FBref      _common.py:250

[5/11/2026 2:40:35 PM] WARNING  c:\Users\LouisHarle\OneDrive -                                  ]8;id=13005563;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=13005564;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py#230\230]8;;\
                                Quadratic\Bureau\Stage_VM\.venv\Lib\site-packages\soccerdata\fb                    
                                ref.py:104: UserWarning: You are trying to scrape data for all                     
                                of the Big 5 European leagues. This can be done more                               
                                efficiently by setting leagues='Big 5 European Leagues                             
                                Combined'.                                                                         
                                  warnings.warn(                                                                   
                                                                                                                   

  Extraction : standard...
     17113 lignes | (pas de xG dans ce stat_type)
  Extraction : keeper...
     1226 lignes | (pas de xG dans ce stat_type)
  Extraction : shooting...
     17113 lignes | (pas de xG dans ce stat_type)
  Extraction : playing_time...
     21071 lignes | (pas de xG dans ce stat_type)
  Extraction : misc...
     17113 lignes | (pas de xG dans ce stat_type)
  Fusion 'keeper' : 40 colonnes
  Fusion 'shooting' : 51 colonnes
  Fusion 'playing_time' : 65 colonnes
  Fusion 'misc' : 75 colonnes

Colonnes xG dans le dataset final : []
Dataset final : 17113 lignes x 75 colonnes


In [27]:
# Charger le dataset de base
df_base = pd.read_csv(r'..\data\soccerdata\data_soccerdata.csv', sep=',', encoding='utf-8-sig')


df_final = merge_fbref_understat(
    df_base, 
    df_xg, 
    output_path=r'..\data\soccerdata\data_final_soccerdata.csv'
)

Dédoublonnage Understat : 16601 -> 13841 lignes.
Fusion terminée : 17113 lignes et 80 colonnes.
Taux de correspondance xG (Coverage) : 50.8%
Fichier sauvegardé sous : ..\data\soccerdata\data_final_soccerdata.csv


## 3. Transfermarkt : les données de blessure

In [ ]:
# Chargement des données de blessure des joueurs

df_blessures = run_full_injury_scraping(
    annee_debut, 
    annee_fin, 
    leagues,
    output_file="../data/dataset_blessures.csv",
    max_threads=12
)

--- Étape 1 : Cartographie historique (2020-2025) ---
  Analyse de la saison 2020/2021...
  Analyse de la saison 2021/2022...
  Analyse de la saison 2022/2023...
  Analyse de la saison 2023/2024...
  Analyse de la saison 2024/2025...
  Analyse de la saison 2025/2026...
--- Étape 2 : Scraping des blessures (5505 joueurs) ---
--- Étape 3 : Sauvegarde des données ---
Terminé ! 3506 lignes enregistrées dans : ../data/dataset_blessures_2.csv


,Nom,Club_Moment_Blessure,Saison,Blessure,Debut,Fin,Jours,Matchs_Manques
0,#24 \n ...,FC Fulham,25/26,Problèmes de genou,28 déc. 2025,16 janv. 2026,20 jours,4
1,Joseba Zaldua,Real Sociedad,20/21,Irritation de l'os du pubis,8 sept. 2020,29 sept. 2020,22 jours,3
2,#37 \n ...,Hellas Verona,25/26,Blessure à l'épaule,26 avr. 2026,,17 jours,2
3,#37 \n ...,Hellas Verona,25/26,Lésion du muscle fléchisseur de la jambe,1 mars 2026,13 avr. 2026,44 jours,5
4,#37 \n ...,Hellas Verona,25/26,Problèmes musculaires,13 janv. 2026,13 févr. 2026,32 jours,5


## 3. football-data.co.uk

Ce site collecte les résultats détaillés des matchs de football (scores, statistiques de match) ainsi que les **cotes de paris sportifs** provenant des principaux bookmakers mondiaux.


La valeur d'un joueur n'est pas isolée ; elle est fortement influencée par la **force de son équipe** et la difficulté de son championnat. Les cotes de paris sportifs servent ici d'indicateur pour mesurer la dominance d'un club et le prestige d'une rencontre. Intégrer ces données permet de pondérer les performances individuelles par le niveau collectif de l'équipe du joueur.


Les données de Football-Data.co.uk apportent une dimension contextuelle essentielle à l’analyse de la valeur marchande. Elles permettent de relier les performances individuelles à l’environnement collectif dans lequel évolue le joueur.

Les résultats des matchs et les statistiques collectives permettent d’évaluer la dynamique d’une équipe, son niveau de domination et sa capacité à créer des occasions. Un joueur évoluant dans une équipe performante et régulière bénéficie généralement d’une meilleure valorisation.

Les données disciplinaires apportent des informations sur le style de jeu et l’intensité collective, tandis que les cotes des bookmakers reflètent la perception du marché sur la force d’un club. Elles permettent ainsi de mesurer la réputation et la compétitivité de l’environnement dans lequel évolue le joueur, deux facteurs qui influencent directement sa valeur marchande.


On peut alors supposer que la valeur marchande évolue en fonction de ces différentes caractéristiques collectives. À performance égale, un joueur évoluant dans une équipe dont la cote moyenne est inférieure à 1.50 aura une valeur marchande supérieure par rapport à un joueur d'une équipe "outsider". De plus, une corrélation positive est attendue entre le ratio de victoires et la hausse de la valeur marchande lors de la mise à jour suivante sur Transfermarkt. Le championnat pèse également dans la balance : des recherches bibliographiques nous ont mené à supposer l'hypothèse d'une inflation structurelle des prix pour les joueurs de Premier League par rapport aux autres ligues du Big 5, même pour des équipes de bas de tableau.

In [30]:
# Appel de la fonction de téléchargement
download_football_data_datasets(annee_debut, annee_fin, leagues)

[5/11/2026 2:43:30 PM] WARNING  c:\Users\LouisHarle\OneDrive -                                  ]8;id=13005569;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=13005570;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py#230\230]8;;\
                                Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\imports                    
                                .py:26: PerformanceWarning: DataFrame is highly fragmented.                        
                                This is usually the result of calling `frame.insert` many                          
                                times, which has poor performance.  Consider joining all                           
                                columns at once using pd.concat(axis=1) instead. To get a                          
                                de-fragmented frame, use `newframe = frame.copy()`                                 
                                                                                                                   
                                                                                                                   

                       WARNING  c:\Users\LouisHarle\OneDrive -                                  ]8;id=13005575;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=13005576;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py#230\230]8;;\
                                Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\imports                    
                                .py:26: PerformanceWarning: DataFrame is highly fragmented.                        
                                This is usually the result of calling `frame.insert` many                          
                                times, which has poor performance.  Consider joining all                           
                                columns at once using pd.concat(axis=1) instead. To get a                          
                                de-fragmented frame, use `newframe = frame.copy()`                                 
                                                                                                                   
                                                                                                                   

[5/11/2026 2:43:31 PM] WARNING  c:\Users\LouisHarle\OneDrive -                                  ]8;id=13005581;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=13005582;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py#230\230]8;;\
                                Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\imports                    
                                .py:26: PerformanceWarning: DataFrame is highly fragmented.                        
                                This is usually the result of calling `frame.insert` many                          
                                times, which has poor performance.  Consider joining all                           
                                columns at once using pd.concat(axis=1) instead. To get a                          
                                de-fragmented frame, use `newframe = frame.copy()`                                 
                                                                                                                   
                                                                                                                   

                       WARNING  c:\Users\LouisHarle\OneDrive -                                  ]8;id=13005587;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=13005588;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py#230\230]8;;\
                                Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\imports                    
                                .py:26: PerformanceWarning: DataFrame is highly fragmented.                        
                                This is usually the result of calling `frame.insert` many                          
                                times, which has poor performance.  Consider joining all                           
                                columns at once using pd.concat(axis=1) instead. To get a                          
                                de-fragmented frame, use `newframe = frame.copy()`                                 
                                                                                                                   
                                                                                                                   

[5/11/2026 2:43:32 PM] WARNING  c:\Users\LouisHarle\OneDrive -                                  ]8;id=13005593;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py\_py_warnings.py]8;;\:]8;id=13005594;file://C:\Users\LouisHarle\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\_py_warnings.py#230\230]8;;\
                                Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\imports                    
                                .py:26: PerformanceWarning: DataFrame is highly fragmented.                        
                                This is usually the result of calling `frame.insert` many                          
                                times, which has poor performance.  Consider joining all                           
                                columns at once using pd.concat(axis=1) instead. To get a                          
                                de-fragmented frame, use `newframe = frame.copy()`                                 
                                                                                                                   
                                                                                                                   

Téléchargement terminé !


Les données de résultats et de cotes pour les 3 dernières saisons du Big 5 sont prêtes. Nous disposons désormais du contexte collectif nécessaire pour pondérer les performances individuelles des joueurs par la force de leur équipe respective.

## Annexe 1 : FBref

FBref fournit des statistiques de performance agrégées par saison, principalement issues des données **Opta**. Contrairement aux données transactionnelles de Transfermarkt, FBref se concentre directement sur le jeu en tant que tel.

En effet, si Transfermarkt donne les valeurs marchandes, FBref donne les **justificatifs de performance** qui expliquent ce prix. Ces statistiques agissent comme des **variables explicatives** fondamentales. Elles permettent de distinguer un joueur efficace par chance d'un joueur créant régulièrement des occasions de haute qualité, ce qui influence directement sa valeur marchande.


Les données de FBref permettent d’analyser la valeur marchande des joueurs à travers leurs performances sportives.

Le temps de jeu renseigne sur la régularité, la disponibilité et l’importance d’un joueur dans son équipe, des éléments directement liés à sa valorisation. Les performances offensives permettent d’évaluer son efficacité et sa capacité à être décisif.

Les indicateurs collectifs mesurent son influence sur les résultats de l’équipe, tandis que les statistiques spécifiques aux gardiens permettent d’évaluer leur fiabilité défensive. Enfin, les données défensives et disciplinaires complètent l’analyse en mettant en évidence l’impact du joueur dans les duels et les phases défensives.


On peut s'attendre à des effets diverses sur la valeur marchandes. Par exemple, les joueur toujours titulaires devraient avoir une valeur marchande supérieure aux autres joueurs, tandis qu'une accumulation de cartons rouges ou de fautes commises sans volume défensif associé pourrait agir comme un malus sur la valeur marchande.

In [ ]:
# Configuration pour utiliser la fonction de téléchargement de données kaggle
DATASET = 'hubertsidorowicz/football-players-stats-2025-2026'
DEST = "../data/fbref_datasets"

# Appel de la fonction de téléchargement
download_kaggle_dataset(DATASET, DEST)

Téléchargement de hubertsidorowicz/football-players-stats-2025-2026 vers ../data/fbref_datasets...
Dataset URL: https://www.kaggle.com/datasets/hubertsidorowicz/football-players-stats-2025-2026
Téléchargement correctement effectué !


Les statistiques de la saison 2025-2026 ont été récupérées. Ces données constituent notre socle de "performance récente".

## Annexe 2 : Statsbomb

StatsBomb propose un accès "Open Data" à une partie de ses bases de données professionnelles. Contrairement aux sources précédentes, il s'agit de **données d'événements**. Chaque ligne représente une action technique précise (passe, tir, tacle, pression) géolocalisée sur le terrain via des coordonnées $(x, y)$.


Cette source est indispensable pour capturer le profil technique du joueur. Là où FBref nous dit qu'un joueur a réussi une passe, StatsBomb nous permet de calculer la difficulté de cette passe (distance, angle, nombre d'adversaires éliminés). Cela permet d'identifier des joueurs dont les statistiques classiques sont modestes mais dont la contribution technique à la progression du ballon est importante, justifiant ainsi des valeurs marchandes élevées ou en devenir.


Les données Statsbomb sont alors plus précises et permettent de percevoir des aspects techniques invisibles par d'autres données. En effet, ces données répertorient sur plusieurs compétitions et plusieurs saisons una analyse technique approfondie (pressions subies, longueurs de passes, localisations $x,y$). On identifie aussi les postes précis occupés par les joueurs sur le terrain en mesurant leur impact tactique, en sachant le contexte de chaque rencontre (adversaire, stade, date).


Nous pouvons supposer que les joueurs affichant un taux de réussite élevé sous pression possèdent une valeur marchande supérieure, car cette compétence est rare et recherchée par les clubs d'élite. De plus, on s'attend à ce que les joueurs capables de réaliser des passes progressives (brisant des lignes) dans le dernier tiers du terrain voient leur valeur augmenter plus rapidement que les joueurs effectuant des passes latérales sécurisées. Enfin, les coordonnées des actions permettront de valider si un joueur s'approche souvent de la surface adverse, augmentant mécaniquement son attractivité financière.

Téléchargons maintenant l'ensemble des données issues de l'Open source de Statsbomb. Les fichiers sont récoltés sous le format .json pour le moment.

In [6]:
# Configuration pour utiliser la fonction de téléchargement de données git
REPO = "https://github.com/statsbomb/open-data"
DEST = "../data/statsbomb_datasets"

# Appel de la fonction de téléchargement
download_github_dataset(REPO, DEST) # type: ignore

Le dossier existe déjà. Vérification des mises à jour...
Les données sont déjà à jour !


Il s'agit maintenant de transformer ces données en fichiers .feather, plus pratique pour nous analyses futures.

In [7]:
# Dossier où on stocke les fichiers finaux
destination = "../data/statsbomb_datasets/data"

In [8]:
# Les compétitions

compile_statsbomb_to_feather(
    json_folder_path="../data/statsbomb_datasets/data", 
    output_folder_path=destination,
    output_name="competitions_statsbomb",
    recursive=False
)

Traitement de 1 fichiers trouvés dans data...
Fusion et sauvegarde...
Terminé ! Fichier : competitions_statsbomb.feather (75 lignes)


In [9]:
# Les lineups

compile_statsbomb_to_feather(
    json_folder_path="../data/statsbomb_datasets/data/lineups", 
    output_folder_path=destination,
    output_name="all_lineups",
    record_path=['lineup'],
    meta=['team_name', 'team_id'],
    recursive = False
)

Traitement de 3464 fichiers trouvés dans lineups...
Fusion et sauvegarde...
Terminé ! Fichier : all_lineups.feather (131901 lignes)


In [10]:
# Les events

# On ne garde que les colonnes essentielles car il y a trop d'informations dans ces fichiers.
cols_events = [
    'match_id', 'id', 'index', 'period', 'timestamp', 'minute', 'second', 
    'type.name', 'team.name', 'player.name', 'position.name', 
    'location', 'duration', 'under_pressure', 'pass.end_location', 
    'pass.outcome.name', 'shot.statsbomb_xg', 'shot.outcome.name'
]

compile_statsbomb_to_feather(
    json_folder_path="../data/statsbomb_datasets/data/events", 
    output_folder_path=destination,
    output_name="all_events",
    columns_to_keep=cols_events,
    recursive = False
)

Traitement de 3464 fichiers trouvés dans events...
Fusion et sauvegarde...
Terminé ! Fichier : all_events.feather (12188949 lignes)


In [11]:
# Les matches

# Les colonnes essentielles pour les matches
cols_matches = [
    'match_id', 'match_date', 'kick_off', 'competition.competition_name', 
    'season.season_name', 'home_team.home_team_name', 'away_team.away_team_name', 
    'home_score', 'away_score'
]


compile_statsbomb_to_feather(
    json_folder_path="../data/statsbomb_datasets/data/matches", 
    output_folder_path=destination,
    output_name="all_matches",
    columns_to_keep=cols_matches,
    recursive = True
)

Traitement de 75 fichiers trouvés dans matches...
Fusion et sauvegarde...
Terminé ! Fichier : all_matches.feather (3464 lignes)


Les fichiers JSON volumineux ont été transformés en format .feather pour optimiser la vitesse de lecture et l'usage de la mémoire RAM.